# Medical Insurance Cost Prediction - Modeling

In this notebook, we will implement and train four different models on our processed dataset:

1. Linear Regression
2. Decision Tree Regressor
3. Random Forest Regressor
4. Neural Network

For each model, we will:
- Load the processed data
- Split the data into training and testing sets
- Train the model
- Make predictions
- Save the model for later evaluation


In [8]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.neural_network import MLPRegressor
# from sklearn.metrics import mean_squared_error, r2_score
# import joblib
# import os

# # Create models directory if it doesn't exist
# os.makedirs('../models', exist_ok=True)

# # Load the processed data
# processed_data = pd.read_csv('../data/processed/insurance_processed.csv')

# # Separate features and target
# y = processed_data['charges']
# X = processed_data.drop('charges', axis=1)


# # Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# print("Features:", X.columns.tolist())
# print("\nTraining set shape:", X_train.shape)
# print("Testing set shape:", X_test.shape)


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Load the processed data
processed_data_train = pd.read_csv('../data/processed/insurance_train_scaled.csv')
processed_data_test = pd.read_csv('../data/processed/insurance_test_scaled.csv')

# Separate features and target
y_train = processed_data_train['charges']
X_train = processed_data_train.drop('charges', axis=1)

y_test = processed_data_test['charges']
X_test = processed_data_test.drop('charges', axis=1)








## 1. Linear Regression

First, we'll train a simple linear regression model as our baseline.


In [15]:
# Initialize and train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
lr_train_pred = lr_model.predict(X_train)
lr_test_pred = lr_model.predict(X_test)

# Calculate metrics
lr_train_mse = mean_squared_error(y_train, lr_train_pred)
lr_test_mse = mean_squared_error(y_test, lr_test_pred)
lr_train_r2 = r2_score(y_train, lr_train_pred)
lr_test_r2 = r2_score(y_test, lr_test_pred)

print("Linear Regression Results:")
print(f"Training MSE: {lr_train_mse:.2f}")
print(f"Testing MSE: {lr_test_mse:.2f}")
print(f"Training R2: {lr_train_r2:.4f}")
print(f"Testing R2: {lr_test_r2:.4f}")

# Save the model
joblib.dump(lr_model, '../models/linear_regression.joblib')


Linear Regression Results:
Training MSE: 36979860.90
Testing MSE: 35478020.68
Training R2: 0.7299
Testing R2: 0.8069


['../models/linear_regression.joblib']

## 2. Decision Tree Regressor

Next, we'll train a Decision Tree model, which can capture non-linear relationships in the data.


In [11]:
# #Initialize and train Decision Tree model
# # dt_model = DecisionTreeRegressor(max_depth=5,random_state=42)
# dt_model = DecisionTreeRegressor(random_state=42)

# dt_model.fit(X_train, y_train)

# # Make predictions
# dt_train_pred = dt_model.predict(X_train)
# dt_test_pred = dt_model.predict(X_test)

# # Calculate metrics
# dt_train_mse = mean_squared_error(y_train, dt_train_pred)
# dt_test_mse = mean_squared_error(y_test, dt_test_pred)
# dt_train_r2 = r2_score(y_train, dt_train_pred)
# dt_test_r2 = r2_score(y_test, dt_test_pred)

# print("Decision Tree Results:")
# print(f"Training MSE: {dt_train_mse:.2f}")
# print(f"Testing MSE: {dt_test_mse:.2f}")
# print(f"Training R2: {dt_train_r2:.4f}")
# print(f"Testing R2: {dt_test_r2:.4f}")

# # Save the model
# joblib.dump(dt_model, '../models/decision_tree.joblib')


In [16]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# --------------------------------------------------
# ۱. تعریف فضای جستجوی پارامترها (Hyperparameter Grid)
# --------------------------------------------------
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': [None, 'sqrt', 'log2']
}

# --------------------------------------------------
# ۲. راه‌اندازی GridSearchCV
# استفاده از ۵-Fold Cross Validation برای یافتن بهترین ترکیب
# --------------------------------------------------
print("Starting Hyperparameter Tuning for Decision Tree...")
grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,               # تقسیم داده‌های آموزش به ۵ قسمت برای اعتبار‌سنجی چرخشی
    scoring='r2',       # بهینه‌سازی بر اساس معیار R2
    n_jobs=-1,          # استفاده از تمام هسته‌های CPU برای سرعت بیشتر
    verbose=1           # نمایش روند پیشرفت
)

# --------------------------------------------------
# ۳. آموزش مدل (جستجو فقط روی داده‌های آموزش انجام می‌شود)
# --------------------------------------------------
grid_search.fit(X_train, y_train)

# استخراج بهترین مدل یافته شده
dt_model = grid_search.best_estimator_

print("\nBest Parameters Found:")
print(grid_search.best_params_)

# --------------------------------------------------
# ۴. پیش‌بینی با مدل بهینه شده
# --------------------------------------------------
dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

# --------------------------------------------------
# ۵. محاسبه و نمایش معیارها
# --------------------------------------------------
dt_train_mse = mean_squared_error(y_train, dt_train_pred)
dt_test_mse = mean_squared_error(y_test, dt_test_pred)
dt_train_r2 = r2_score(y_train, dt_train_pred)
dt_test_r2 = r2_score(y_test, dt_test_pred)



print("-" * 30)
print("Decision Tree Results (Optimized with GridSearch):")
print(f"Training MSE: {dt_train_mse:.2f}")

print(f"Testing MSE: {dt_test_mse:.2f}")
print(f"Training R2: {dt_train_r2:.4f}")
print(f"Testing R2: {dt_test_r2:.4f}")
print("-" * 30)

# --------------------------------------------------
# ۶. ذخیره مدل نهایی
# --------------------------------------------------
joblib.dump(dt_model, '../models/decision_tree.joblib')
print("Model saved successfully!")

Starting Hyperparameter Tuning for Decision Tree...
Fitting 5 folds for each of 240 candidates, totalling 1200 fits

Best Parameters Found:
{'max_depth': 5, 'max_features': None, 'min_samples_leaf': 6, 'min_samples_split': 2}
------------------------------
Decision Tree Results (Optimized with GridSearch):
Training MSE: 18226172.14
Testing MSE: 19198240.41
Training R2: 0.8669
Testing R2: 0.8955
------------------------------
Model saved successfully!


## 3. Random Forest Regressor

Now we'll train a Random Forest model, which is an ensemble of decision trees and often provides better generalization.


In [ ]:
# # Initialize and train Random Forest model
# rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
# rf_model.fit(X_train, y_train)

# # Make predictions
# rf_train_pred = rf_model.predict(X_train)
# rf_test_pred = rf_model.predict(X_test)

# # Calculate metrics
# rf_train_mse = mean_squared_error(y_train, rf_train_pred)
# rf_test_mse = mean_squared_error(y_test, rf_test_pred)
# rf_train_r2 = r2_score(y_train, rf_train_pred)
# rf_test_r2 = r2_score(y_test, rf_test_pred)

# print("Random Forest Results:")
# print(f"Training MSE: {rf_train_mse:.2f}")
# print(f"Testing MSE: {rf_test_mse:.2f}")
# print(f"Training R2: {rf_train_r2:.4f}")
# print(f"Testing R2: {rf_test_r2:.4f}")

# # Save the model
# joblib.dump(rf_model, '../models/random_forest.joblib')


In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

# --------------------------------------------------
# ۱. تعریف فضای جستجو (Hyperparameter Grid)
# --------------------------------------------------
param_grid_rf = {
    'n_estimators': [100, 200, 300],          # تعداد درخت‌های جنگل
    'max_depth': [5, 10, 15, None],           # حداکثر عمق هر درخت
    'min_samples_leaf': [1, 2, 4],            # حداقل نمونه در هر برگ برای جلوگیری از اورفیت
   'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None],   # تعداد ویژگی‌هایی که در هر گره بررسی می‌شوند
}

# --------------------------------------------------
# ۲. راه‌اندازی GridSearchCV برای رندوم فارست
# --------------------------------------------------
print("Starting Hyperparameter Tuning for Random Forest... (This might take a few minutes)")
grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    cv=5,               # اعتبار‌سنجی ۵-مرحله‌ای
    scoring='r2',
    n_jobs=-1,          # استفاده از تمام توان پردازنده
    verbose=1
)

# --------------------------------------------------
# ۳. اجرای عملیات جستجو روی داده‌های آموزشی
# --------------------------------------------------
grid_search_rf.fit(X_train, y_train)

# استخراج بهترین مدل
best_rf_model = grid_search_rf.best_estimator_

print("\nBest Parameters found for Random Forest:")
print(grid_search_rf.best_params_)

# --------------------------------------------------
# ۴. پیش‌بینی و محاسبه متریک‌ها با مدل بهینه
# --------------------------------------------------
rf_train_pred = best_rf_model.predict(X_train)
rf_test_pred = best_rf_model.predict(X_test)

rf_train_mse = mean_squared_error(y_train, rf_train_pred)
rf_test_mse = mean_squared_error(y_test, rf_test_pred)
rf_train_r2 = r2_score(y_train, rf_train_pred)
rf_test_r2 = r2_score(y_test, rf_test_pred)
rf_test_mae = mean_absolute_error(y_test, rf_test_pred)
rf_train_mae = mean_absolute_error(y_train, rf_train_pred)

print("-" * 30)
print("Random Forest Results (Optimized):")
print(f"Training MSE: {rf_train_mse:.2f}")
print(f"Testing MSE: {rf_test_mse:.2f}")
print(f"Training MAE: {rf_train_mae:.2f}")
print(f"Testing MAE: {rf_test_mae:.2f}")
print(f"Optimized Training R2: {rf_train_r2:.4f}")
print(f"Optimized Testing R2: {rf_test_r2:.4f}")
print("-" * 30)

# --------------------------------------------------
# ۵. ذخیره مدل بهینه شده
# --------------------------------------------------
joblib.dump(best_rf_model, '../models/random_forest.joblib')
print("Optimized Random Forest model saved!")

Starting Hyperparameter Tuning for Random Forest... (This might take a few minutes)
Fitting 5 folds for each of 324 candidates, totalling 1620 fits

Best Parameters found for Random Forest:
{'max_depth': 5, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
------------------------------
Random Forest Results (Optimized):
Training MSE: 17063122.12
Testing MSE: 17658485.08
Training MAE: 2256.25
Testing MAE: 2364.68
Optimized Training R2: 0.8754
Optimized Testing R2: 0.9039
------------------------------
Optimized Random Forest model saved!


## 4. Neural Network

Finally, we'll train a Neural Network model using MLPRegressor from scikit-learn.


In [ ]:
# # Initialize and train Neural Network model
# nn_model = MLPRegressor(
#     hidden_layer_sizes=(16),  # One hidden layer
#     activation='relu',
#     solver='adam',
#     max_iter=1000,
#     alpha=0.01,
#     random_state=42
# )
# nn_model.fit(X_train, y_train)

# # Make predictions
# nn_train_pred = nn_model.predict(X_train)
# nn_test_pred = nn_model.predict(X_test)

# # Calculate metrics
# nn_train_mse = mean_squared_error(y_train, nn_train_pred)
# nn_test_mse = mean_squared_error(y_test, nn_test_pred)
# nn_train_r2 = r2_score(y_train, nn_train_pred)
# nn_test_r2 = r2_score(y_test, nn_test_pred)

# print("Neural Network Results:")
# print(f"Training MSE: {nn_train_mse:.2f}")
# print(f"Testing MSE: {nn_test_mse:.2f}")
# print(f"Training R2: {nn_train_r2:.4f}")
# print(f"Testing R2: {nn_test_r2:.4f}")

# # Save the model
# joblib.dump(nn_model, '../models/neural_network.joblib')


In [ ]:
# from sklearn.neural_network import MLPRegressor
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import mean_squared_error, r2_score
# import joblib

# # --------------------------------------------------
# # ۱. تعریف فضای جستجو (Hyperparameter Grid)
# # --------------------------------------------------
# # param_grid_nn = {
# #     # تست لایه‌های مختلف: ۱۶ نورون، ۳۲ نورون، و ترکیب (۱۶، ۸)
# #     'hidden_layer_sizes': [(16,), (32,), (16, 8), (32, 16)], 
# #     'activation': ['relu', 'tanh'],
# #     'alpha': [0.0001, 0.001, 0.01], # پارامتر جریمه برای کنترل اورفیت
# #     'learning_rate_init': [0.001, 0.01 , 0.05],
# #     'solver': ['adam', 'lbfgs']
# # }
# param_grid_nn = {
#     'hidden_layer_sizes': [(16,8), (32,16), (64,32)],
#     'activation': ['relu', 'tanh'],
#     'solver': ['adam'],
#     'alpha': [0.001, 0.01],
#     'learning_rate_init': [0.001, 0.01]
# }
# # --------------------------------------------------
# # ۲. راه‌اندازی GridSearchCV
# # --------------------------------------------------
# print("Starting Hyperparameter Tuning for Neural Network...")
# grid_search_nn = GridSearchCV(
#     estimator=MLPRegressor( max_iter=2000, random_state=42),
#     param_grid=param_grid_nn,
#     cv=5,               # اعتبار‌سنجی ۵-مرحله‌ای
#     scoring='r2',
#     n_jobs=-1,          # استفاده از تمام توان پردازنده
#     verbose=1
# )

# # --------------------------------------------------
# # ۳. آموزش و جستجو (فقط روی داده‌های آموزش)
# # --------------------------------------------------
# grid_search_nn.fit(X_train_scaled, y_train)

# # استخراج بهترین مدل یافته شده
# nn_model = grid_search_nn.best_estimator_

# print("\nBest Parameters Found:")
# print(grid_search_nn.best_params_)

# # --------------------------------------------------
# # ۴. پیش‌بینی با مدل بهینه شده
# # --------------------------------------------------
# nn_train_pred = nn_model.predict(X_train_scaled)
# nn_test_pred = nn_model.predict(X_test_scaled)

# # --------------------------------------------------
# # ۵. محاسبه و نمایش معیارها
# # --------------------------------------------------
# nn_train_mse = mean_squared_error(y_train, nn_train_pred)
# nn_test_mse = mean_squared_error(y_test, nn_test_pred)
# nn_train_r2 = r2_score(y_train, nn_train_pred)
# nn_test_r2 = r2_score(y_test, nn_test_pred)

# import warnings
# from sklearn.exceptions import ConvergenceWarning

# warnings.filterwarnings("ignore", category=ConvergenceWarning)

# print("-" * 30)
# print("Neural Network Results (Optimized):")
# print(f"Training MSE: {nn_train_mse:.2f}")
# print(f"Testing MSE: {nn_test_mse:.2f}")
# print(f"Optimized Training R2: {nn_train_r2:.4f}")
# print(f"Optimized Testing R2: {nn_test_r2:.4f}")
# print("-" * 30)

# # --------------------------------------------------
# # ۶. ذخیره مدل نهایی
# # --------------------------------------------------
# joblib.dump(nn_model, '../models/neural_network.joblib')
# print("Optimized Neural Network model saved successfully!")

## 5. Cross-Validation Evaluation

In [18]:
from sklearn.model_selection import cross_val_score

models = [
    ('Linear Regression', lr_model),
    ('Decision Tree', dt_model),
    ('Random Forest', best_rf_model)
]

print("K-Fold Cross-Validation Results (R2 Score):")
print("-" * 50)

cv_results = []
for name, model in models:
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    cv_results.append({
        'Model': name,
        'CV Mean R2': cv_scores.mean(),
        'CV Std': cv_scores.std()
    })
    
    print(f"{name}:")
    print(f"  Mean R2: {cv_scores.mean():.4f}")
    print(f"  Standard Deviation: {cv_scores.std():.4f}")
    print("-" * 30)

# تبدیل نتایج به DataFrame برای نمایش بهتر
cv_df = pd.DataFrame(cv_results)
display(cv_df)

K-Fold Cross-Validation Results (R2 Score):
--------------------------------------------------
Linear Regression:
  Mean R2: 0.7258
  Standard Deviation: 0.0253
------------------------------
Decision Tree:
  Mean R2: 0.8286
  Standard Deviation: 0.0182
------------------------------
Random Forest:
  Mean R2: 0.8436
  Standard Deviation: 0.0190
------------------------------


,Model,CV Mean R2,CV Std
0,Linear Regression,0.725757,0.025277
1,Decision Tree,0.828631,0.018170
2,Random Forest,0.843576,0.019041


## Model Comparison

Let's compare the performance of all models side by side.


In [21]:
# Create a comparison dataframe
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'Training MSE': [lr_train_mse, dt_train_mse, rf_train_mse],
    'Testing MSE': [lr_test_mse, dt_test_mse, rf_test_mse],
    'Training R2': [lr_train_r2, dt_train_r2, rf_train_r2],
    'Testing R2': [lr_test_r2, dt_test_r2, rf_test_r2]
})

#########print("Overfitting Gap:", dt_train_r2 - dt_test_r2)

print("Model Performance Comparison:")
print(results.to_string(index=False))


Model Performance Comparison:
            Model  Training MSE  Testing MSE  Training R2  Testing R2
Linear Regression  3.697986e+07 3.547802e+07     0.729906    0.806929
    Decision Tree  1.822617e+07 1.919824e+07     0.866879    0.895523
    Random Forest  1.706312e+07 1.765849e+07     0.875374    0.903903
